In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv


In [2]:
!pip uninstall -y torchao -q

In [3]:
!pip install -U torchao -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 30.0 MB/s eta 0:00:00


In [4]:
import os
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()
hf_token = user_secrets.get_secret("HF_TOKEN")
os.environ["HF_TOKEN"] = hf_token

from huggingface_hub import login
login(token=hf_token)

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


In [5]:
"""
Milestone 4 - Smart MCQ Solver
Basically turning the prompt+5options rows into something AutoModelForMultipleChoice
can actually chew on, then doing a tiny LoRA finetune just to see the pipeline works
end to end before scaling it up on the full train set later.

Ran this on Kaggle (T4), should also run fine on CPU just slower for the finetune part.
"""

import os
import torch
import pandas as pd
import torch.nn.functional as F
from transformers import (
    AutoTokenizer,
    AutoModelForMultipleChoice,
    TrainingArguments,
    Trainer,
)
from datasets import Dataset
from peft import LoraConfig, get_peft_model, TaskType

torch.manual_seed(42)

# on kaggle the comp data usually sits under /kaggle/input/<comp-name>/
# keeping a fallback here so it also runs locally against the copies I have
CANDIDATE_PATHS = [
    "/kaggle/input/smart-mcq-solver-challenge/train.csv",
    "train.csv",
    "/mnt/project/train_7.csv",
]
train_path = '/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv'
df = pd.read_csv(train_path)
print("loaded train set:", df.shape, "from", train_path)

OPTION_COLS = ["A", "B", "C", "D", "E"]
device = "cuda" if torch.cuda.is_available() else "cpu"

# ---------------------------------------------------------------
# Q1 - label encoding, letters to ints
# ---------------------------------------------------------------
label2id = {letter: idx for idx, letter in enumerate(OPTION_COLS)}
df["label"] = df["answer"].map(label2id)

row150_label = int(df.loc[150, "label"])
print(f"Q1 -> encoded label for row 150: {row150_label}")

# ---------------------------------------------------------------
# Q2 - prompt + option formatting, checking string length for row 0 option B
# ---------------------------------------------------------------
def make_pair(prompt, option):
    return str(prompt) + " [SEP] " + str(option)

row0 = df.iloc[0]
option_b_text = make_pair(row0["prompt"], row0["B"])
print(f"Q2 -> length of formatted option B string (row 0): {len(option_b_text)}")

# ---------------------------------------------------------------
# tokenizer setup, reused for every question below
# ---------------------------------------------------------------
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")


def build_mcq_batch(rows, tok=tokenizer, max_length=128):
    """
    Flattens each row's 5 options into one long list of (prompt, option) pairs,
    tokenizes them all together, then reshapes back to (num_rows, 5, seq_len).
    This is basically the same trick the HF multiple-choice example notebook uses.
    """
    first_sentences = []
    second_sentences = []
    for _, r in rows.iterrows():
        for col in OPTION_COLS:
            first_sentences.append(str(r["prompt"]))
            second_sentences.append(str(r[col]))

    tokenized = tok(
        first_sentences,
        second_sentences,
        padding="max_length",
        truncation=True,
        max_length=max_length,
        return_tensors="pt",
    )

    n_rows = len(rows)
    reshaped = {k: v.view(n_rows, 5, max_length) for k, v in tokenized.items()}
    return reshaped


# ---------------------------------------------------------------
# Q3 - single row tokenized, check second dim of reshaped input_ids
# ---------------------------------------------------------------
single_row_encoded = build_mcq_batch(df.iloc[[0]], max_length=128)
print(f"Q3 -> input_ids shape: {tuple(single_row_encoded['input_ids'].shape)}")
print(f"Q3 -> second dimension (num choices): {single_row_encoded['input_ids'].shape[1]}")

# ---------------------------------------------------------------
# Q4 - first 16 rows, total token positions in the tensor
# ---------------------------------------------------------------
batch16_encoded = build_mcq_batch(df.iloc[:16], max_length=128)
total_positions = batch16_encoded["input_ids"].numel()
print(f"Q4 -> input_ids shape: {tuple(batch16_encoded['input_ids'].shape)}")
print(f"Q4 -> total token positions: {total_positions}")

# ---------------------------------------------------------------
# Q5 - load the MCQ model, run row 0 through it, look at logits shape
# ---------------------------------------------------------------
mcq_model = AutoModelForMultipleChoice.from_pretrained("bert-base-uncased").to(device)
mcq_model.eval()

with torch.no_grad():
    out = mcq_model(
        input_ids=single_row_encoded["input_ids"].to(device),
        attention_mask=single_row_encoded["attention_mask"].to(device),
    )
print(f"Q5 -> logits shape: {tuple(out.logits.shape)}")
print(f"Q5 -> number of logits for one question: {out.logits.shape[1]}")

# ---------------------------------------------------------------
# Q6 - same row but with the label passed in, check loss tensor dims
# ---------------------------------------------------------------
row0_label_tensor = torch.tensor([row150_label if False else int(df.loc[0, "label"])]).to(device)
# (kept the ternary above just so it's obvious this is row 0's label, not row 150's)

with torch.no_grad():
    out_with_loss = mcq_model(
        input_ids=single_row_encoded["input_ids"].to(device),
        attention_mask=single_row_encoded["attention_mask"].to(device),
        labels=row0_label_tensor,
    )
print(f"Q6 -> loss value: {out_with_loss.loss.item():.4f}")
print(f"Q6 -> loss tensor ndim: {out_with_loss.loss.dim()}")

# ---------------------------------------------------------------
# Q7 - wrap the model with LoRA, count trainable params
# ---------------------------------------------------------------
lora_cfg = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["query", "value"],
    lora_dropout=0.1,
    bias="none",
    task_type=TaskType.SEQ_CLS,
)
lora_model = get_peft_model(mcq_model, lora_cfg).to(device)
trainable_params = sum(p.numel() for p in lora_model.parameters() if p.requires_grad)
print(f"Q7 -> trainable params after LoRA: {trainable_params}")
lora_model.print_trainable_parameters()

# ---------------------------------------------------------------
# Q8 - turn first 100 rows into a HF Dataset the Trainer can consume
# ---------------------------------------------------------------
subset_100 = df.iloc[:100].reset_index(drop=True)
enc_100 = build_mcq_batch(subset_100, max_length=128)

hf_dataset_100 = Dataset.from_dict(
    {
        "input_ids": enc_100["input_ids"].tolist(),
        "attention_mask": enc_100["attention_mask"].tolist(),
        "labels": subset_100["label"].tolist(),
    }
)
first_item_shape = torch.tensor(hf_dataset_100[0]["input_ids"]).shape
print(f"Q8 -> first item input_ids shape: {tuple(first_item_shape)}")
print(f"Q8 -> tokenized choices stored: {first_item_shape[0]}")


def collate_mcq(features):
    # everything's already padded to max_length, so this is really just stacking
    batch = {
        "input_ids": torch.tensor([f["input_ids"] for f in features]),
        "attention_mask": torch.tensor([f["attention_mask"] for f in features]),
        "labels": torch.tensor([f["labels"] for f in features]),
    }
    return batch


# ---------------------------------------------------------------
# Q9 - tiny finetune run, 32 rows, max_length 64, only 4 steps
# ---------------------------------------------------------------
subset_32 = df.iloc[:32].reset_index(drop=True)
enc_32 = build_mcq_batch(subset_32, max_length=64)
hf_dataset_32 = Dataset.from_dict(
    {
        "input_ids": enc_32["input_ids"].tolist(),
        "attention_mask": enc_32["attention_mask"].tolist(),
        "labels": subset_32["label"].tolist(),
    }
)

# fresh LoRA model for the actual training run (the one above already saw a forward pass,
# doesn't matter much for a 4-step toy run but keeping it clean)
base_for_training = AutoModelForMultipleChoice.from_pretrained("bert-base-uncased").to(device)
train_model = get_peft_model(base_for_training, lora_cfg).to(device)

training_args = TrainingArguments(
    output_dir="./mcq_lora_tiny_run",
    per_device_train_batch_size=4,
    gradient_accumulation_steps=1,
    max_steps=4,
    logging_steps=1,
    save_strategy="no",
    report_to="none",
    remove_unused_columns=False,
)

trainer = Trainer(
    model=train_model,
    args=training_args,
    train_dataset=hf_dataset_32,
    data_collator=collate_mcq,
)
train_result = trainer.train()
print(f"Q9 -> final global_step: {trainer.state.global_step}")

# ---------------------------------------------------------------
# Q10 - probability for option E on row 0, using the just-finetuned model
# ---------------------------------------------------------------
train_model.eval()
row0_encoded_64 = build_mcq_batch(df.iloc[[0]], max_length=64)
with torch.no_grad():
    final_out = train_model(
        input_ids=row0_encoded_64["input_ids"].to(device),
        attention_mask=row0_encoded_64["attention_mask"].to(device),
    )
probs = F.softmax(final_out.logits, dim=-1).squeeze()
prob_e = probs[4].item()
print(f"Q10 -> softmax probabilities (A-E): {[round(p, 4) for p in probs.tolist()]}")
print(f"Q10 -> probability assigned to option E: {round(prob_e, 4)}")

print("\ndone - answers for Q1 to Q10 printed above")

Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.10.0+cpu).


loaded train set: (2000, 8) from /kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
Q1 -> encoded label for row 150: 2
Q2 -> length of formatted option B string (row 0): 407


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

...

Q5 -> logits shape: (1, 5)
Q5 -> number of logits for one question: 5
Q6 -> loss value: 1.6243
Q6 -> loss tensor ndim: 0
Q7 -> trainable params after LoRA: 295681
trainable params: 295,681 || all params: 109,778,690 || trainable%: 0.2693
Q8 -> first item input_ids shape: (5, 128)
Q8 -> tokenized choices stored: 5


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForMultipleChoice LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
/usr/local/lib/python3.12/dist-packages/torch

Step,Training Loss
1,1.599842
2,1.628842
3,1.569262
4,1.614611


Q9 -> final global_step: 4
Q10 -> softmax probabilities (A-E): [0.2049, 0.2066, 0.203, 0.1944, 0.1911]
Q10 -> probability assigned to option E: 0.1911

done - answers for Q1 to Q10 printed above


In [6]:
# ---------------------------------------------------------------
# quick summary of all 10 answers, easier to screenshot for the form
# ---------------------------------------------------------------
summary = {
    "Q1 (row 150 label)": row150_label,
    "Q2 (option B string length, row 0)": len(option_b_text),
    "Q3 (second dim of input_ids)": single_row_encoded["input_ids"].shape[1],
    "Q4 (total token positions, 16 rows)": total_positions,
    "Q5 (num logits per question)": out.logits.shape[1],
    "Q6 (loss tensor ndim)": out_with_loss.loss.dim(),
    "Q7 (trainable params after LoRA)": trainable_params,
    "Q8 (choices stored in input_ids)": first_item_shape[0],
    "Q9 (final global_step)": trainer.state.global_step,
    "Q10 (prob of option E)": round(prob_e, 4),
}

print("\n" + "=" * 40)
print("SUMMARY - all 10 answers")
print("=" * 40)
for question, value in summary.items():
    print(f"{question}: {value}")


SUMMARY - all 10 answers
Q1 (row 150 label): 2
Q2 (option B string length, row 0): 407
Q3 (second dim of input_ids): 5
Q4 (total token positions, 16 rows): 10240
Q5 (num logits per question): 5
Q6 (loss tensor ndim): 0
Q7 (trainable params after LoRA): 295681
Q8 (choices stored in input_ids): 5
Q9 (final global_step): 4
Q10 (prob of option E): 0.1911
